# House Price Prediction with Linear Regression

## Project objective
Build and evaluate a regression model that predicts house prices from property characteristics. This notebook follows a reproducible workflow: data inspection, exploratory analysis, feature reasoning, preprocessing, train/test split, model training, evaluation, diagnostics, coefficient interpretation, and a regularised-model comparison.

**Dataset:** `House Price Prediction Dataset.csv` (2,000 rows; 10 columns at initial inspection).

**Tech stack:** Python, pandas, NumPy, matplotlib, seaborn, scikit-learn.

> Run all cells from top to bottom. The notebook automatically searches for the CSV beside the notebook and, as a fallback, in `/mnt/data/`.

## 1. Import libraries and load the dataset

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, r2_score

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

candidates = [
    Path("House Price Prediction Dataset.csv"),
    Path("/mnt/data/House Price Prediction Dataset.csv")
]
DATA_PATH = next((p for p in candidates if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Place 'House Price Prediction Dataset.csv' beside this notebook "
        "or update DATA_PATH to its location."
    )

df = pd.read_csv(DATA_PATH)
print("Dataset:", DATA_PATH)
print("Shape:", df.shape)
display(df.head())

## 2. Exploratory Data Analysis (EDA)
Inspect structure, missingness, duplicates, descriptive statistics, and the target distribution before modelling.

In [ ]:
print("Dataset information:")
df.info()

print("\\nMissing values:")
display(df.isna().sum().to_frame("missing_count"))

print("Exact duplicate rows:", int(df.duplicated().sum()))
print("Duplicate IDs:", int(df["Id"].duplicated().sum()) if "Id" in df.columns else "Id column not present")

print("\\nDescriptive statistics:")
display(df.describe(include="all").T)

In [ ]:
TARGET = "Price"
if TARGET not in df.columns:
    raise KeyError(f"Expected target column '{TARGET}' was not found.")

plt.figure(figsize=(9, 5))
sns.histplot(df[TARGET].dropna(), kde=True, bins=30)
plt.title("Distribution of House Prices")
plt.xlabel("Price")
plt.ylabel("Number of properties")
plt.tight_layout()
plt.show()

print("Target skewness:", round(df[TARGET].skew(), 3))

In [ ]:
# Numeric feature distributions and boxplots
numeric_for_eda = df.select_dtypes(include=np.number).columns.drop(TARGET, errors="ignore")
df[numeric_for_eda].hist(figsize=(14, 9), bins=25, edgecolor="black")
plt.suptitle("Numeric Feature Distributions", y=1.02)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
sns.boxplot(x=df[TARGET])
plt.title("House Price Boxplot")
plt.tight_layout()
plt.show()

## 3. Feature selection — reasoning

The target is `Price`, which is the value the model must predict.

Potential predictors:
- **Area:** larger properties may generally command higher prices.
- **Bedrooms and Bathrooms:** represent capacity and convenience.
- **Floors:** may capture property size/layout.
- **YearBuilt:** can represent property age and condition-related value.
- **Location:** neighbourhood/location can strongly influence market value; it is categorical.
- **Condition:** property condition may influence buyer willingness to pay.
- **Garage:** availability of a garage may affect value.

**Exclude `Id`:** it is an identifier, not a meaningful property characteristic. Including it could let the model learn an arbitrary row/order pattern.

We will use all listed property characteristics initially, then interpret coefficients and test model performance. Correlation is useful for detecting linear association, but it does not prove causation and can miss non-linear relationships.

## 4. Correlation analysis
The first heatmap uses numeric columns. The second encodes categorical predictors with one-hot encoding for an exploratory feature-to-price correlation view. This analysis is performed before the train/test split only as EDA; all learned preprocessing for the predictive model is fitted on training data inside a pipeline.

In [ ]:
numeric_df = df.select_dtypes(include=np.number)
plt.figure(figsize=(10, 7))
sns.heatmap(numeric_df.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Heatmap — Numeric Columns")
plt.tight_layout()
plt.show()

# Exploratory encoded correlation: excludes identifier and target from predictors.
eda_features = df.drop(columns=["Price", "Id"], errors="ignore").copy()
eda_encoded = pd.get_dummies(eda_features, drop_first=True, dtype=int)
eda_encoded["Price"] = df["Price"]
price_corr = eda_encoded.corr(numeric_only=True)["Price"].drop("Price").sort_values()

plt.figure(figsize=(9, max(5, 0.32 * len(price_corr))))
price_corr.plot(kind="barh")
plt.title("Encoded Feature Correlation with House Price")
plt.xlabel("Pearson correlation")
plt.tight_layout()
plt.show()

display(price_corr.to_frame("correlation_with_price"))

## 5. Prepare features and split the data

We separate predictors (`X`) from the target (`y`). The identifier is excluded. The test set is held out and is not used to fit imputers, encoders, scalers, or the model.

- **80% training data:** used to learn preprocessing and model parameters.
- **20% test data:** used for final evaluation on unseen rows.
- `random_state=42`: makes the split reproducible.

In [ ]:
TARGET = "Price"
DROP_COLUMNS = ["Id", TARGET]
X = df.drop(columns=DROP_COLUMNS, errors="ignore")
y = pd.to_numeric(df[TARGET], errors="coerce")

# Remove rows with missing target values: supervised learning needs a known label.
valid_target = y.notna()
X = X.loc[valid_target].copy()
y = y.loc[valid_target].copy()

numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

print("Numeric predictors:", numeric_features)
print("Categorical predictors:", categorical_features)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

## 6. Preprocessing: missing values and One-Hot Encoding

A `ColumnTransformer` applies separate preprocessing:
- Numeric columns: median imputation, then standardisation.
- Categorical columns: most-frequent imputation, then One-Hot Encoding.

The preprocessing is placed inside a scikit-learn `Pipeline`. This is important because the imputation statistics, category vocabulary, and scaling parameters are learned from **training data only**, preventing test-data leakage.

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", drop="first"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

## 7. Train Linear Regression

In [ ]:
linear_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

linear_model.fit(X_train, y_train)
y_pred = linear_model.predict(X_test)

print("Linear Regression training complete.")
print("Number of test predictions:", len(y_pred))

## 8. Model evaluation: MSE, RMSE, and R²
MSE and RMSE measure prediction error in price units (RMSE); lower is better. R² measures the proportion of target variance explained by the model on this test set; closer to 1 is generally better. A negative R² means the model performs worse than predicting the test-set mean.

In [ ]:
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

metrics = pd.DataFrame({
    "Metric": ["MSE", "RMSE", "R²"],
    "Value": [mse, rmse, r2]
})
display(metrics)
print(f"MSE : {mse:,.2f}")
print(f"RMSE: {rmse:,.2f}")
print(f"R²  : {r2:.4f}")

## 9. Actual vs. predicted prices
Points close to the diagonal reference line indicate predictions close to the actual prices. Large deviations indicate larger prediction errors.

In [ ]:
plt.figure(figsize=(7, 7))
sns.scatterplot(x=y_test, y=y_pred, alpha=0.65)
lower = min(float(y_test.min()), float(np.min(y_pred)))
upper = max(float(y_test.max()), float(np.max(y_pred)))
plt.plot([lower, upper], [lower, upper], linestyle="--", label="Perfect prediction")
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Actual vs. Predicted House Prices")
plt.legend()
plt.tight_layout()
plt.show()

## 10. Residual analysis
A residual is actual price minus predicted price. Ideally, residuals are scattered around zero without a clear curve, funnel, or other systematic pattern. A pattern can indicate non-linearity, unequal error variance, or omitted variables.

In [ ]:
residuals = y_test - y_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(x=y_pred, y=residuals, alpha=0.65, ax=axes[0])
axes[0].axhline(0, color="red", linestyle="--")
axes[0].set_title("Residuals vs. Predicted Price")
axes[0].set_xlabel("Predicted Price")
axes[0].set_ylabel("Residual (Actual - Predicted)")

sns.histplot(residuals, kde=True, ax=axes[1])
axes[1].axvline(0, color="red", linestyle="--")
axes[1].set_title("Residual Distribution")
axes[1].set_xlabel("Residual")
plt.tight_layout()
plt.show()

## 11. Coefficient analysis

Linear regression coefficients describe the model's fitted direction and size of association, holding other encoded features constant.

Because numeric predictors were standardised, their coefficients represent the estimated price change for a one-standard-deviation increase in that feature. One-hot categorical coefficients compare a category with its dropped reference category. Coefficients are associations within this fitted model, not causal effects.

In [ ]:
fitted_preprocessor = linear_model.named_steps["preprocessor"]
feature_names = fitted_preprocessor.get_feature_names_out()
coefficients = linear_model.named_steps["model"].coef_

coef_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients
})
coef_df["Absolute_Coefficient"] = coef_df["Coefficient"].abs()
coef_df = coef_df.sort_values("Absolute_Coefficient", ascending=False)

print("Largest positive coefficients:")
display(coef_df.sort_values("Coefficient", ascending=False).head(10))
print("Largest negative coefficients:")
display(coef_df.sort_values("Coefficient", ascending=True).head(10))

top_coef = coef_df.head(15).sort_values("Coefficient")
plt.figure(figsize=(10, 7))
sns.barplot(data=top_coef, x="Coefficient", y="Feature")
plt.axvline(0, color="black", linewidth=1)
plt.title("Largest Linear Regression Coefficients")
plt.tight_layout()
plt.show()

## 12. Bonus: compare Linear Regression with Ridge Regression
Ridge adds an L2 penalty to discourage very large coefficients. It can improve generalisation when predictors are correlated or the model is overfitting. The `alpha` value controls regularisation strength; here we use a simple baseline value of 1.0. A stronger project could tune alpha with cross-validation using training data only.

In [ ]:
ridge_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", Ridge(alpha=1.0))
])
ridge_model.fit(X_train, y_train)
ridge_pred = ridge_model.predict(X_test)

comparison = pd.DataFrame([
    {
        "Model": "Linear Regression",
        "MSE": mean_squared_error(y_test, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
        "R²": r2_score(y_test, y_pred)
    },
    {
        "Model": "Ridge Regression (alpha=1.0)",
        "MSE": mean_squared_error(y_test, ridge_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, ridge_pred)),
        "R²": r2_score(y_test, ridge_pred)
    }
])
display(comparison)

## 13. Final conclusion

After running the notebook, write a short conclusion using the actual metrics printed above:

1. State the test-set RMSE and R² for Linear Regression.
2. Compare Ridge with Linear Regression and identify which performed better on this split.
3. Refer to the actual-vs-predicted plot and residual plot to describe model fit and any visible patterns.
4. Name the strongest positive and negative coefficients, remembering that one-hot coefficients are relative to reference categories.
5. Explain limitations: this is a single random split, linear regression assumes additive linear relationships, and the dataset may not include important real-world factors such as neighbourhood-level prices, property amenities, and market conditions.

**Do not claim the model is accurate or production-ready based only on one score.** Validate on more data and use cross-validation/hyperparameter tuning before deployment.

## 14. Save model results (optional)

This cell exports the held-out predictions for further inspection. It does not save or deploy the trained model.

In [ ]:
predictions = pd.DataFrame({
    "Actual_Price": y_test.to_numpy(),
    "Predicted_Price": y_pred,
    "Residual": residuals.to_numpy()
})
predictions.to_csv("house_price_test_predictions.csv", index=False)
print("Saved: house_price_test_predictions.csv")
display(predictions.head(10))